# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [4]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issues with loans, based on the complaints provided, appear to include errors with loan balances and account information, mishandling of payments (such as misapplied payments or inability to pay towards the principal), problems with loan transfers and lack of proper notification, discrepancies in reported loan status and balance, and difficulties with repayment plans or forgiveness. \n\nOverall, a prevalent issue is mismanagement or miscommunication by loan servicers, leading to incorrect balances, unauthorized transfers, and improper handling of payments and account status.'

In [13]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, at least one complaint indicates that it did not get handled in a timely manner. Specifically, the complaint filed with Maximus Federal Services, Inc. (Complaint ID: 13160766), received on 04/24/25, was marked as "Timely response?": "Yes," but the consumer reported that their issue was unresolved and that they had been working on the problem for nearly 18 months with no resolution. Additionally, the complaint with MOHELA (Complaint ID: 12709087), received on 03/28/25, was marked as "Timely response?": "No," indicating it was not handled in a timely manner.\n\nThus, yes, there were complaints that did not get handled in a timely manner.'

In [14]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, as reflected in the complaints:\n\n1. **Accumulation of Interest During Forbearance or Deferment:** Many borrowers lost ground because interest continued to accrue during forbearance or deferment periods, making it difficult to pay down the principal and increasing the total amount owed over time.\n\n2. **Inability to Afford Increased Payments:** When attempting to increase monthly payments to pay off loans faster, borrowers often found they couldn’t afford the higher payments due to their financial situation, which included basic living expenses like food and transportation.\n\n3. **Poor Communication and Lack of Transparency:** Borrowers were often not properly notified about loan transfer dates, repayment resumption, or changes in servicers. Some were unaware of the exact terms or when payments should restart, leading to missed payments and subsequent delinquency.\n\n4. **Mismanagement and Lack of Clear Information:** Compl

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [15]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [16]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [17]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans, particularly federal student loans, appears to be problems related to dealing with lenders or servicers. Specifically, frequent complaints include:\n\n- Disputes over fees charged and lack of clear explanations.\n- Difficulty in applying payments correctly, often resulting in payments going toward interest rather than principal.\n- Receiving incorrect or bad information about loan balances, terms, or history.\n- Issues with loan repayment terms, such as overly long repayment periods or difficulty navigating the loan management system.\n\nThese issues indicate that a significant common problem is poor communication, lack of transparency, and difficulties in managing or understanding loan details with servicers.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all the complaints listed have been responded to with a "Closed with explanation" status and are marked as "Timely response?": "Yes." Therefore, it appears that any complaints that were handled in a timely manner were addressed appropriately. There is no indication of complaints that were not handled in a timely manner.'

In [19]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for several reasons, including issues with their payment plans, miscommunication or lack of communication from the loan servicers, and problems with the handling of their payments. For example, some individuals experienced repeated problems with their payments being reversed or not processed correctly, often due to errors with the servicers like Aidvantage. Others were unaware of changes to their loan status because they did not receive proper notifications, especially when loans were transferred or when autopayments were discontinued without their knowledge. Additionally, some borrowers were misled into incorrect forbearance options or were not informed about their overdue status, leading to negative impacts on their credit scores. Overall, these issues are often related to poor communication, mismanagement, or errors by the loan servicers, which hindered borrowers' ability to successfully repay their loans."

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

</div>

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">
### Answer:

- During the activity, BM25 actually performed worse than the other embeddings
-  An example query where BM25 is a better embedding is for key word search or for exact string: "What can I reference in Clip Attach FMEA document JPV-12100-EB01"

</div>

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [20]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [21]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, it appears that a common issue with loans, particularly student loans, involves problems related to the handling and communication about the loans. The most frequently mentioned issues include errors or discrepancies in loan balances, misapplied payments, lack of clear or correct information about the loan details, and improper handling or transfer of the loans. Additionally, issues related to poor communication, incorrect or misleading information from servicers, and violations of privacy laws are common complaints.\n\nIn summary, the most common issue with loans from this context seems to be **mismanagement of loan information and poor communication from lenders or servicers, leading to errors, confusion, and disputes over balances and loan details.**'

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that at least one complaint was not handled in a timely manner. Specifically, the complaint regarding the student loan issues submitted to Maximus Federal Services, Inc. has been open since an unspecified date ("XXXX") and has remained unresolved for nearly 18 months, despite multiple requests for resolution. The complainant explicitly states it has been nearly 18 months without resolution. \n\nHowever, for the complaints related to EdFinancial Services, the responses were marked as "closed with explanation" and indicated as "Yes" for timely response, suggesting those were handled in a timely manner.\n\nSo, the complaint about Maximus Federal Services, Inc. did not get resolved in a timely manner.'

In [25]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Lack of awareness or understanding: Many borrowers were not informed or did not realize that their financial aid would need to be repaid. They often received bad information from financial aid officers or were unaware of the repayment obligations.\n\n2. Poor communication and notification: Borrowers experienced inadequate notification from loan servicers about when payments were due, changes in loan ownership, or the need to set up payment plans. Some did not receive proper alerts or updates, leading to missed or late payments.\n\n3. Accumulation of interest during deferment or forbearance: When borrowers entered forbearance or deferment, interest continued to accrue, increasing the total amount owed even if they temporarily paused payments.\n\n4. Difficulties managing payments: The available payment options, such as forbearance or deferment, often resulted in interest piling up, making it harder to pay off th

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [26]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [27]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [28]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints data and analysis, the most common issues with student loans are related to:\n\n- Errors or inaccuracies in loan balances and account information.\n- Mismanagement or mishandling of payments, often leading to late payments or erroneous reporting.\n- Problems with repayment plans, including inability to apply extra payments to principal or pay off loans quicker.\n- Unnotified loan transfers or changes in servicers, causing confusion and missed payments.\n- Wrongful reporting of delinquencies or defaults to credit bureaus.\n- Difficulty in resolving disputes, correcting account errors, or obtaining accurate information due to poor communication or disorganized servicing.\n- Unauthorized or poorly communicated changes in loan status, including ending deferments or misclassification of loan types.\n- Inadequate notice of loan transfer, leading to missed payments and credit impacts.\n\nOverall, the most common theme is mismanagement and lack of clear, trans

In [29]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, according to the provided complaints, several complaints indicate that they were not handled in a timely manner. For example:\n\n- One complaint (row 67) from 04/14/25 cites that the response was "Yes" for timely response, but the issue remains unresolved after over 2-3 weeks.\n- Several complaints involving delays or lack of response (rows 95, 616, 611, 66) mention that complaints or issues have been ongoing for over a year or longer without resolution.\n- Notably, complaint ID 12823876 mentions efforts over nearly two and a half years with continued delays and unresolved issues.\n- Additionally, multiple complaints report that the complaint either was not responded to or took longer than expected, with some being unresolved for over 18 months.\n\nTherefore, based on the data, it appears that several complaints did not get handled in a timely manner.'

In [30]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often fail to pay back their loans due to a variety of issues, including mismanagement by loan servicers, lack of proper communication or notification about payment requirements, legal discrepancies or errors in their account status, and systemic failures within the student loan servicing system. Additionally, some borrowers are unaware of their repayment obligations or are misled into forbearance or consolidation options that can increase their debt through interest capitalization, making repayment more difficult. Many also face financial hardships, such as unemployment, medical issues, or unexpected expenses, which hinder their ability to repay loans. Furthermore, errors in reporting or legal discrepancies in account information can contribute to missed payments and default.\n\nIn summary, failure to pay back loans can stem from administrative errors, lack of transparent information, systemic issues, or personal financial hardships.'

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

</div>

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Answer:

Reformulations of a user query can improve recall by: covering vocabulary mismatch, capturing different tokenization patterns, and expanding semantic search.

</div>

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [31]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [32]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [33]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [34]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [35]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [36]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans appear to be related to errors and misconduct by loan servicers, including incorrect or misleading information on credit reports, disputes over loan balances and interest rates, misapplied payments, wrongful denials of payment plans, and issues with loan transfers and legitimacy of debt. \n\nTherefore, the most common issue with loans seems to be problems arising from loan servicing misconduct, errors, and mismanagement that negatively impact borrowers.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that several complaints were not handled in a timely manner. Specifically, the complaints related to student loan servicing by MOHELA (Complaint IDs: 12709087 and 12935889) explicitly state "No" for timely response, indicating they were not handled promptly. Additionally, the complaint involving Aidvantage (Complaint ID: 12950199) was marked as "Yes" for timely response, meaning it was addressed in a timely manner.\n\nTherefore, yes, some complaints, notably those against MOHELA, did not get handled in a timely manner.'

In [38]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of inadequate communication, mismanagement, and unforeseen financial hardships. For example, some individuals experienced issues with loan servicers resuming payments prematurely, without proper notification or reevaluation based on grace periods. Others faced severe financial difficulties after attending institutions that misrepresented their value, concealed financial problems, or ultimately closed, making it difficult to secure employment and repay loans. Additionally, there were cases where incorrect reporting, lack of proper notification about payment obligations, or disputes over the legitimacy of debts contributed to the inability to repay loans. Overall, these issues reflect a mix of insufficient information, institutional misconduct, and personal financial hardship.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [39]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [40]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans tend to revolve around dealing with lenders or servicers, such as receiving bad or inaccurate information, mishandling of account details, unauthorized transfers, poor communication, or errors in loan balances and interest calculations. Many complaints also involve difficulty in obtaining clear, transparent information about loan terms, interest accrual, payments, and account status.\n\nIn particular, the recurring theme is that consumers frequently experience mishandling and miscommunication from loan servicers, which leads to inaccuracies in loan balances, interest charges, or default status, and often results in financial hardship or credit score impacts. \n\nTherefore, the most common issue appears to be **"Dealing with your lender or servicer," especially involving misinformation, mismanagement, or lack of transparency about loan details and account handling.**'

In [42]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, based on the provided complaints, some complaints indicate that complaints did not get handled in a timely manner. For example:\n\n- Complaint ID 12935889 from MOHELA received on 04/11/25 was marked "No" for timely response, indicating it was not handled promptly.\n- Complaint ID 12475477 from MOHELA received on 03/25/25 was marked "No" for timely response, also indicating delays.\n- Multiple other complaints mention delays, extended wait times (sometimes over hours), or lack of response despite follow-up efforts.\n\nTherefore, it appears that there were complaints that were not handled in a timely manner.'

In [43]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to a combination of systemic issues, miscommunication, and adverse economic circumstances. Based on the provided complaints, the main reasons include:\n\n1. **Lack of Clear Communication and Notification**: Many borrowers were not adequately notified about when their repayment was to begin, changes in loan servicers, or the transfer of their loans. Lack of proper notices led to missed payments and misunderstandings.\n\n2. **Mismanagement and Errors by Servicers**: Complaints highlight errors such as incorrect loan balances, misapplied payments, wrongful delinquency reporting, and illegal or unverified debt collection practices. Servicers sometimes failed to provide proper documentation or verification of debts.\n\n3. **Inappropriate or Misleading Repayment Options**: Borrowers were often steered into forbearances or deferred payments without fully understanding that interest would continue to accrue and compound, increasing the total

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [44]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [45]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [46]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [47]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [48]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [49]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided data, the most common issues with loans appear to be related to problems with loan servicing and reporting. Specifically, frequent complaints involve:\n\n- Trouble with repayment plans and payment handling (e.g., auto-debit issues, payment amount disputes)\n- Incorrect or improper reporting of loan status, such as accounts being inaccurately marked as delinquent or in default\n- Lack of communication and transparency from loan servicers regarding loan changes or statuses\n- Unauthorized or illegal use of borrower data and privacy breaches\n- Difficulties with loan forgiveness or discharge processes\n\nWhile multiple issues are highlighted, problems related to loan servicing, including mismanagement of payments and inaccurate reporting, seem to be most prevalent.'

In [53]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, according to the provided data, several complaints indicate issues with handling in a timely manner. Notably, complaints about Nelnet, Inc. mention delays or lack of response despite multiple sent letters and attempts to resolve issues. For example, one complaint states that despite acknowledgment of receipt of certified mail and detailed misconduct, Nelnet "never responded to the CM, nor provided any answers to the questions raised." \n\nAdditionally, the complaint about AidVantage\'s autopay setup was handled within the same day, indicating a timely response there, but other complaints involving Nelnet and similar servicers suggest delays or lack of handling in a timely fashion.\n\nTherefore, based on the documented complaints, some issues did not receive timely handling, especially those involving Nelnet, Inc.'

In [51]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"Based on the provided context, people failed to pay back their loans for various reasons, including:\n\n- Lack of transparency and difficulty obtaining accurate or timely information from lenders or servicers, which can hinder borrowers' ability to stay informed about their loan status or available options.\n- Procedural issues such as errors in documentation or communication, leading to misunderstandings about loan status or repayment obligations.\n- Disputes over the legitimacy or accuracy of reported debts, which can result in negative credit reports despite the borrower’s claims of never being in default.\n- Allegations of unfair or illegal practices by loan servicers, such as improper reporting, failure to verify debts, or unauthorized access to personal information, which can complicate repayment efforts.\n- Technical or administrative errors that prevent payments from being processed correctly, leading to defaults or defaults being reported incorrectly.\n- Borrowers’ disputes o

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

</div>

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Answer:

I would lower the chunk size and chunk overlap. Short sentences on higher chunk size would likely mean that the chunk will capture non similar context.

</div>

# 🤝 Breakout Room Part #2

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against each other. 
You can use the loans or bills dataset.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

</div>

##### HINTS:

- LangSmith provides detailed information about latency and cost.

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Analysis & Observations:

</div>